## Config

In [1]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 50        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 256
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

TRAIN_SPLIT = 0.9   # split by books
MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000
CKPT_DIR = 'checkpoints'
SAVE_EVERY_EPOCH = True
SAVE_EVERY_STEPS = 200
# ------------------------------------

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## Load vocab + metadata

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 26401
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 49


## Dataset (windowed next-token prediction)

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, 'r', encoding='utf-8') as f:
        s = f.read().strip()
    if not s:
        return []
    return list(map(int, s.split()))

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_books(books: List[BookData], train_split: float) -> Tuple[List[BookData], List[BookData]]:
    books = books[:]
    random.shuffle(books)
    cut = int(len(books) * train_split)
    return books[:cut], books[cut:]

def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

class CachedWindowDataset(Dataset):
    def __init__(self, books, seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens: List[torch.Tensor] = []
        self.samples: List[Tuple[int, int]] = []

        for b in books:
            ids = load_ids_file(b.ids_path)
            if len(ids) < seq_len + 2:
                continue

            t = torch.tensor(ids, dtype=torch.int32)  # compact in RAM
            bi = len(self.book_tokens)
            self.book_tokens.append(t)

            max_start = t.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)  # convert only this slice
        return chunk[:-1], chunk[1:]

books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_books, val_books = split_books(books, TRAIN_SPLIT)
train_ds = CachedWindowDataset(train_books, SEQ_LEN, STRIDE)
val_ds = CachedWindowDataset(val_books, SEQ_LEN, STRIDE)

print(f'Books: total={len(books)} train={len(train_books)} val={len(val_books)}')
print(f'Samples: train={len(train_ds):,} val={len(val_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # IMPORTANT for Windows notebook stability
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: total=49 train=44 val=5
Samples: train=292,267 val=32,418


## Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)          # (B,T,E)
        out, _ = self.lstm(e)    # (B,T,H)
        out = self.drop(out)
        logits = self.fc(out)    # (B,T,V)
        return logits

model = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_id=pad_id,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

use_amp = (DEVICE == "cuda")
scaler = GradScaler("cuda", enabled=use_amp)

sum(p.numel() for p in model.parameters())

23980577

## Sampling utilities

In [5]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

prompt = [bos_id] if bos_id is not None else []
print(decode(sample_ids(model, prompt, max_new_tokens=40, temperature=1.0, top_k=50)))

<bos> abrupt v sighvat suitable bereaved wash wash essayed governor poetry pretended emitted chisels charged mug mishosha's granary guffawed chuckling extraneous money trilling waits muanza money nephews siva disappointed barrow nobleman's comedians brings detachment thro procure stall corpses promptly regained tattercoats


## Train + evaluate

In [6]:
def run_eval() -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl

global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen_tokens = 0

    for step, (x, y) in enumerate(train_loader, start=1):
        global_step += 1
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            logits = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        scaler.scale(loss).backward()

        # if you clip grads, you must unscale first
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)   # <-- this is the optimizer step
        scaler.update()

        # logging
        mask = (y != pad_id)
        tok = int(mask.sum().item())
        running += float(loss.item()) * tok
        seen_tokens += tok

        if global_step % SAVE_EVERY_STEPS == 0:
            avg = running / max(1, seen_tokens)
            print(f"epoch {epoch} step {step} train_loss={avg:.4f} train_ppl={math.exp(min(20.0, avg)):.2f}")
            ckpt_path = os.path.join(CKPT_DIR, f"lstm_step{global_step}.pt")
            torch.save({"model_state": model.state_dict(), "vocab": vocab}, ckpt_path)
            print(f"[saved] {ckpt_path}", flush=True)

        if step >= STEPS_PER_EPOCH:
            break

    train_loss = running / max(1, seen_tokens)
    val_loss, val_ppl = run_eval()
    print(f"\n== epoch {epoch} done ==")
    print(f"train_loss={train_loss:.4f} train_ppl={math.exp(min(20.0, train_loss)):.2f}")
    print(f"val_loss={val_loss:.4f}   val_ppl={val_ppl:.2f}")

    # sample (make sure sample_ids sets model.eval() inside, or do it here)
    prompt = [bos_id] if bos_id is not None else []
    out = sample_ids(model, prompt, max_new_tokens=120, temperature=1.0, top_k=50)
    print("\n[sample]")
    print(decode(out))

    if SAVE_EVERY_EPOCH:
        ckpt_path = os.path.join(CKPT_DIR, f"lstm_lm_epoch{epoch}.pt")
        torch.save({
            "model_state": model.state_dict(),
            "vocab": vocab,
            "config": {
                "SEQ_LEN": SEQ_LEN,
                "STRIDE": STRIDE,
                "EMB_DIM": EMB_DIM,
                "HIDDEN_SIZE": HIDDEN_SIZE,
                "NUM_LAYERS": NUM_LAYERS,
                "DROPOUT": DROPOUT,
            }
        }, ckpt_path)
        print(f"[saved] {ckpt_path}\n")

epoch 1 step 200 train_loss=6.5366 train_ppl=689.93
[saved] checkpoints\lstm_step200.pt
epoch 1 step 400 train_loss=6.2166 train_ppl=500.98
[saved] checkpoints\lstm_step400.pt
epoch 1 step 600 train_loss=5.9866 train_ppl=398.05
[saved] checkpoints\lstm_step600.pt
epoch 1 step 800 train_loss=5.8259 train_ppl=338.96
[saved] checkpoints\lstm_step800.pt
epoch 1 step 1000 train_loss=5.7076 train_ppl=301.16
[saved] checkpoints\lstm_step1000.pt

== epoch 1 done ==
train_loss=5.6395 train_ppl=281.33
val_loss=4.9995   val_ppl=148.34

[sample]
<bos> and, and he is nothing in these life, sir lancelot and to be a little man of my way, " i do not see my own land! ' we am your son, if i would be sure of that are you! " " the young man? " i am sorry of any one of my father, " said the king. i can't ride from a great, a young man, " but the poor man has been to the <unk> of the <unk> and a moment and in his little place and i heard that it can be sorry at an room before the great man, and his brother


## Tests

In [7]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

In [14]:
prompt = "the older fairies stood all in a group , saying loudly"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

the older fairies stood all in a group, saying loudly: " my son, i am ready, the youngest knight, who must come to take, then for the sake of the king whom you trusted to me. " then the king ordered his son to take the gardener's daughter to the palace, and there was no more beautiful and beautiful one. she would ask, and the prince said: " you must not be able to come near him. " so she began to look at her eyes. she had hardly ceased to hide her window; and every furrow went round her, and her eyes she was as sweet as ever, and one day she passed through


In [24]:
prompt = "once upon a time"
prompt_ids = encode_prompt(prompt)
out = sample_ids(model, prompt_ids, temperature=0.9, top_k=50)
print(decode(out))

once upon a time the king became the bride of the dead-goddess. " and now that she was a woman of the lake, she would have thought it was too much for her to be as the owner of the box on her own. it was a very pretty baby, and she always was all the fairies. she was always the old woman to herself. what do you suppose, and he is, she would not refuse to help her, since she could not tell her how to make her happy. but the old woman, seeing that she had just left the same time to tell her that she had a beautiful
